In [1]:
from langgraph.graph import StateGraph, END
from Agent_state import agent_state
from nodes import agent_node
# from langgraph.prebuilt import tools_condition
from react_agent_runnable import tools
from langchain.messages import AIMessage

In [2]:
flow=StateGraph(agent_state)

In [3]:
def should_continue(state):
    lastAIMessage=state.get("output")
    if isinstance(lastAIMessage, AIMessage) and hasattr(lastAIMessage,"tool_calls") and len(lastAIMessage.tool_calls) > 0:
        return "tools"
    else:
        return END

In [4]:
def tool_node(state):
    lastAIMessage=state.get("output")
    tool_call=lastAIMessage.tool_calls
    new_intermediate_step=[]
    for calls in tool_call:
        tool_name=calls["name"]
        tool_arg=calls["args"]

        tool_to_run=next((t for t in tools if tool_name==t.name),None)

        if tool_to_run:
            tool_result=tool_to_run.invoke(tool_arg)

            new_intermediate_step.append((tool_name,tool_result))
    return {"intermediatestep": [(tool_name, str(tool_result))]}


In [5]:
flow.add_node("agent",agent_node)
flow.add_node("tools",tool_node)
flow.add_edge("tools","agent")
flow.add_conditional_edges("agent",should_continue)
flow.set_entry_point("agent")

In [6]:
agent=flow.compile()

In [7]:
agent.invoke({
    "input":"whats the current time?",
    "intermediatestep":[],
    "output":None,
})

{'input': 'whats the current time?',
 'intermediatestep': [],
 'output': AIMessage(content='**Answer (including the reasoning that led to it):**\n\n- The conversation’s system prompt supplies the current date and time for reference: **2026‑06‑18\u202f04:02:57\u202fUTC**.  \n- No external tools are needed because the required information is already present in the prompt.  \n- Therefore, the current time is:\n\n**2026‑06‑18\u202f04:02:57\u202fUTC**.', additional_kwargs={'reasoning_content': "\nThe current time is provided in the initial instructions. I will use that information to answer the user's question.\n\nThe current UTC time is 2026-06-18 04:02:57 UTC.\n\n\nThe current time is 2026-06-18 04:02:57 UTC."}, response_metadata={'token_usage': {'completion_tokens': 219, 'prompt_tokens': 1212, 'total_tokens': 1431, 'completion_time': 0.471719, 'completion_tokens_details': None, 'prompt_time': 0.061797, 'prompt_tokens_details': None, 'queue_time': 1.304169, 'total_time': 0.533516}, 'model